# Optional Notebook 5D — Downloading ABIDE fMRI and selecting an atlas region with Nilearn

**Quantitative Methods in Neuroscience · Master in Neuroscience · University of Geneva**  
**Optional extension — not assessed**

This notebook demonstrates how Nilearn can:

1. download one quality-checked, preprocessed ABIDE resting-state fMRI participant;
2. inspect a four-dimensional NIfTI image;
3. calculate and plot a mean functional image;
4. download a cortical atlas;
5. select one Default Mode Network parcel;
6. overlay that parcel on the participant's image;
7. extract and plot the parcel's resting-state time series.

> This is a visualization and data-access example. A single participant is not sufficient for group inference or clinical conclusions.

## 0 · Before running

Use the dedicated **Python (qmn-optional)** environment created from the included `environment.yml`.

The first execution downloads:

- one preprocessed ABIDE functional image;
- the Schaefer 2018 atlas.

The exact download size depends on the selected ABIDE site and file. The files are cached under `notebooks/data/optional_nilearn/` and are reused later.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import nilearn
from nilearn import datasets, image, plotting
from nilearn.maskers import NiftiLabelsMasker

RUN_DOWNLOAD = True

DATA_DIR = Path("data") / "optional_nilearn"
ABIDE_DIR = DATA_DIR / "abide"
ATLAS_DIR = DATA_DIR / "atlases"

print("Python:", sys.version.split()[0])
print("Nilearn:", nilearn.__version__)
print("Cache directory:", DATA_DIR.resolve())

## 1 · Download one preprocessed ABIDE participant

`fetch_abide_pcp()` accesses the **Preprocessed Connectomes Project** version of ABIDE. We request:

- one participant only;
- the CPAC preprocessing pipeline;
- the preprocessed four-dimensional functional image;
- participants who passed the fetcher's quality-check criterion.

This is much lighter than downloading all ABIDE participants.

In [ ]:
if RUN_DOWNLOAD:
    abide = datasets.fetch_abide_pcp(
        data_dir=ABIDE_DIR,
        n_subjects=1,
        pipeline="cpac",
        derivatives=["func_preproc"],
        quality_checked=True,
        verbose=1,
    )

    func_path = Path(abide.func_preproc[0])
    phenotype = pd.DataFrame(abide.phenotypic)

    print("Functional image:", func_path)
    print(f"File size: {func_path.stat().st_size / 1024**2:.1f} MB")
    display(phenotype.head())
else:
    func_path = None
    print("Download skipped in offline-validation mode.")

## 2 · Inspect the four-dimensional fMRI image

A resting-state fMRI file contains one three-dimensional brain volume at each acquired time point. The fourth dimension is therefore time.

In [ ]:
if RUN_DOWNLOAD:
    func_img = image.load_img(func_path)
    shape = func_img.shape
    voxel_sizes = func_img.header.get_zooms()
    repetition_time = float(voxel_sizes[3])

    image_summary = {
        "shape": shape,
        "spatial_voxel_sizes_mm": voxel_sizes[:3],
        "n_time_points": shape[3],
        "TR_seconds": repetition_time,
    }
    display(image_summary)

## 3 · Calculate and plot the mean functional image

A mean image averages across the time dimension. It is not an activation map; it is a convenient anatomical-quality view of the functional data.

In [ ]:
if RUN_DOWNLOAD:
    mean_func_img = image.mean_img(func_img)
    plotting.plot_epi(
        mean_func_img,
        display_mode="ortho",
        cut_coords=(0, -20, 20),
        title="ABIDE participant — mean preprocessed functional image",
        colorbar=True,
    )
    plotting.show()

## 4 · Download the Schaefer 2018 cortical atlas

An atlas divides the brain into labelled regions. We use the 100-parcel, seven-network Schaefer atlas because it is small enough for teaching and includes network names in its labels.

In [ ]:
if RUN_DOWNLOAD:
    atlas = datasets.fetch_atlas_schaefer_2018(
        n_rois=100,
        yeo_networks=7,
        resolution_mm=2,
        data_dir=ATLAS_DIR,
        verbose=1,
    )

    labels = [
        label.decode("utf-8") if isinstance(label, bytes) else str(label)
        for label in atlas.labels
    ]

    print("Atlas image:", atlas.maps)
    print("Number of labels:", len(labels))
    display(pd.Series(labels[:12], name="atlas_label"))

## 5 · Select one Default Mode Network parcel

Instead of choosing a parcel number manually, the code searches the atlas labels for the first parcel containing `Default`. This makes the selection transparent and reproducible.

In [ ]:
if RUN_DOWNLOAD:
    atlas_img = image.load_img(atlas.maps)
    atlas_data = atlas_img.get_fdata()

    # Match labels to the non-background integer values actually stored in the atlas.
    # This avoids assuming whether a particular Nilearn release includes a
    # background entry in ``atlas.labels``.
    parcel_values = np.unique(atlas_data)
    parcel_values = parcel_values[parcel_values != 0].astype(int)

    if len(parcel_values) != len(labels):
        raise RuntimeError(
            "The number of non-background atlas values does not match the labels. "
            f"Values: {len(parcel_values)}, labels: {len(labels)}"
        )

    label_to_value = dict(zip(labels, parcel_values))
    selected_label = next(
        label for label in labels
        if "Default" in label
    )
    selected_value = label_to_value[selected_label]

    print("Selected atlas value:", selected_value)
    print("Selected parcel:", selected_label)

    selected_roi_img = image.new_img_like(
        atlas_img,
        (atlas_data == selected_value).astype(np.int8),
    )

## 6 · Overlay the selected parcel on the participant's mean fMRI image

Nilearn automatically handles image resampling when needed for visualization and masking. The colored region is an anatomical atlas parcel, not a participant-specific statistical result.

In [ ]:
if RUN_DOWNLOAD:
    plotting.plot_roi(
        selected_roi_img,
        bg_img=mean_func_img,
        display_mode="ortho",
        title=f"Selected Schaefer parcel: {selected_label}",
        alpha=0.65,
    )
    plotting.show()

## 7 · Extract the selected parcel's time series

`NiftiLabelsMasker` converts the voxel data inside a labelled region into a region-level time series. Here we additionally:

- remove a linear trend;
- standardize the signal;
- apply a teaching-example 0.01–0.10 Hz temporal band;
- use the repetition time stored in the NIfTI header.

These preprocessing choices must be justified for a real study and should not be copied mechanically.

In [ ]:
if RUN_DOWNLOAD:
    roi_masker = NiftiLabelsMasker(
        labels_img=selected_roi_img,
        detrend=True,
        standardize="zscore_sample",
        low_pass=0.10,
        high_pass=0.01,
        t_r=repetition_time,
        verbose=0,
    )

    roi_time_series = roi_masker.fit_transform(func_img).squeeze()

    time_seconds = np.arange(roi_time_series.size) * repetition_time
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(time_seconds, roi_time_series, linewidth=1)
    ax.axhline(0, linewidth=0.8, linestyle="--")
    ax.set(
        xlabel="Time (seconds)",
        ylabel="Standardized parcel signal",
        title=f"Resting-state time series: {selected_label}",
    )
    fig.tight_layout()
    plt.show()

    print("Extracted time points:", roi_time_series.size)

## What students should remember

- Nilearn dataset fetchers download and cache public neuroimaging data.
- A functional NIfTI image is usually four-dimensional: three spatial dimensions plus time.
- A mean functional image is useful for inspection but is not an activation map.
- Atlases provide reproducible region definitions.
- Maskers transform images into analysis-ready time series.
- One participant and one parcel provide a software demonstration, not a test of autism-related differences.

## References and documentation

- Nilearn documentation: [`fetch_abide_pcp`](https://nilearn.github.io/stable/modules/generated/nilearn.datasets.fetch_abide_pcp.html), [`fetch_atlas_schaefer_2018`](https://nilearn.github.io/stable/modules/generated/nilearn.datasets.fetch_atlas_schaefer_2018.html), and [Nifti maskers](https://nilearn.github.io/stable/modules/maskers.html).
- Di Martino, A. et al. (2014). **“The Autism Brain Imaging Data Exchange: Towards a Large-Scale Evaluation of the Intrinsic Brain Architecture in Autism.”** *Molecular Psychiatry*, 19, 659–667.
- Nielsen, J. A. et al. (2013). **“Multisite Functional Connectivity MRI Classification of Autism: ABIDE Results.”** *Frontiers in Human Neuroscience*, 7, 599.
- Schaefer, A. et al. (2018). **“Local-Global Parcellation of the Human Cerebral Cortex from Intrinsic Functional Connectivity MRI.”** *Cerebral Cortex*, 28(9), 3095–3114.